# Notebook 4b — Frame Extraction & DINOv3 DinoTXT Vision Embedding Pre-computation

Direct parallel to NB4 (DINOv2), with the vision backbone swapped to DINOv3 ViT-L/16.
Everything else — frame extraction, pooling strategy, output format — is identical,
enabling a controlled comparison between DINOv2 and DINOv3 backbones.

**Design rationale:**
- NB4 used `dinov2_vitl14_reg4_dinotxt_tet1280d20h24l` with `encode_image()` per frame,
  mean-pooled across up to 50 frames. This follows the DINOv2 paper (section 7.2) and
  the reference implementation from pierrefdz (GitHub issue #16).
- We apply the identical protocol here using `dinov3_vitl16_dinotxt_tet1280d20h24l`.
  Although the DINOv3 paper (section 6.1.6) evaluates video using an attentive probe
  over patch tokens, that setup does not apply here: we use `encode_image()` which runs
  the full DinoTXT vision tower (backbone → VisionHead → shared 1280-dim space), so
  the output is directly comparable to the DINOv2 embeddings from NB4.
- Keeping the same protocol across both backbones isolates the effect of the backbone
  itself, which is the research question.

**What changes vs NB4:**
- Model: `dinov3_vitl16_dinotxt_tet1280d20h24l` (loaded from DINOv3 repo)
- Weights: requires two separate files — backbone weights + DinoTXT text tower weights
- Transform: DINOv3's `make_classification_eval_transform()` (patch size 16 vs 14)
- Output dir: `embeddings/dinov3/` (separate from `embeddings/dinov2/`)
- SHARED_DIM: verify from model config (expected 1280, same as DINOv2 DinoTXT)

**What stays identical vs NB4:**
- Frame extraction with decord, up to MAX_FRAMES=50 evenly spaced
- `encode_image(normalize=True)` per frame batch
- Mean pooling over time dimension
- Output: `{segment_id}.pt` float32 tensor of shape (SHARED_DIM,)
- Resumable loop — already computed files are skipped
- Failure tracking and clean CSV output

**Prerequisites:**
1. DINOv3 repo cloned: `git clone https://github.com/facebookresearch/dinov3.git`
2. DINOv3 ViT-L/16 backbone weights downloaded (HuggingFace license approval required)
3. DINOv3 DinoTXT weights downloaded (same approval email includes both)

**Reads:** `annotations/segments_verbs_timed.csv`
**Outputs:** `embeddings/dinov3/{segment_id}.pt` — tensor of shape (SHARED_DIM,)

## Setup

In [1]:
import os, sys, json
import torch
import pandas as pd

with open('/home/ubuntu/youcook2-videos/paths.json') as f:
    PATHS = json.load(f)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

DINOV3_REPO = os.path.join(PATHS['repos'], 'dinov3')
if DINOV3_REPO not in sys.path:
    sys.path.insert(0, DINOV3_REPO)
import dinov3
print(f"dinov3 repo on sys.path ✓")

Device : cuda
GPU    : NVIDIA A10
dinov3 repo on sys.path ✓


## Load data

In [2]:
df = pd.read_csv('/home/ubuntu/youcook2-videos/annotations/segments_verbs_timed.csv')

print(f"Total segments: {len(df):,}")
print(df.head(2))

Total segments: 13,029
                             caption     video_id       segment_id  \
0          crush and chop the garlic  --bv0V6ZjWI  --bv0V6ZjWI_000   
1  add oil garlic and salt to a bowl  --bv0V6ZjWI  --bv0V6ZjWI_001   

  extracted_verb                        masked_caption  start_sec  end_sec  \
0          crush            [MASK] and chop the garlic        8.0     23.0   
1            add  [MASK] oil garlic and salt to a bowl       23.0     47.0   

     subset  
0  training  
1  training  


## Load pretrained DINOv3 DinoTXT — freeze everything

Unlike DINOv2 DinoTXT which is loaded via a single `torch.hub.load` with weights
bundled in the hub, DINOv3 DinoTXT requires two separate weight files:
- `backbone_weights`: the DINOv3 ViT-L/16 vision backbone
- `weights`: the DinoTXT text tower + VisionHead (trained on DINOv3 features)

Both are downloaded after accepting the HuggingFace license agreement.
Update the paths below to wherever you saved them.

In [3]:
model, tokenizer = dinov3_vitl16_dinotxt_tet1280d20h24l(
    weights='/home/ubuntu/weights/dinov3_vitl16_dinotxt_vision_head_and_text_encoder-a442d8f5.pth',
    backbone_weights='/home/ubuntu/weights/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth'
)

Downloading: "file:///home/ubuntu/weights/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth" to /home/ubuntu/.cache/torch/hub/checkpoints/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth


100%|██████████| 1.13G/1.13G [00:00<00:00, 1.37GB/s]


Downloading: "file:///home/ubuntu/weights/dinov3_vitl16_dinotxt_vision_head_and_text_encoder-a442d8f5.pth" to /home/ubuntu/.cache/torch/hub/checkpoints/dinov3_vitl16_dinotxt_vision_head_and_text_encoder-a442d8f5.pth


100%|██████████| 2.10G/2.10G [00:01<00:00, 1.31GB/s]


In [2]:
import sys
import inspect

DINOV3_REPO = '/home/ubuntu/youcook2-videos/repos/dinov3'
if DINOV3_REPO not in sys.path:
    sys.path.insert(0, DINOV3_REPO)

from dinov3.hub.dinotxt import dinov3_vitl16_dinotxt_tet1280d20h24l
print(inspect.getsource(dinov3_vitl16_dinotxt_tet1280d20h24l))

def dinov3_vitl16_dinotxt_tet1280d20h24l(
    *,
    pretrained: bool = True,
    weights: Union[DINOTxtWeights, str] = DINOTxtWeights.LVTD2300M,
    backbone_weights: Union[BackboneWeights, str] = BackboneWeights.LVD1689M,
    bpe_path_or_url: str = "https://dl.fbaipublicfiles.com/dinov3/thirdparty/bpe_simple_vocab_16e6.txt.gz",
    check_hash: bool = False,
) -> Tuple[nn.Module, Any]:
    from dinov3.eval.text.dinotxt_model import DINOTxt, DINOTxtConfig
    from dinov3.eval.text.text_transformer import TextTransformer
    from dinov3.eval.text.tokenizer import get_tokenizer

    dinotxt_config = DINOTxtConfig(
        embed_dim=2048,
        vision_model_freeze_backbone=True,
        vision_model_train_img_size=224,
        vision_model_use_class_token=True,
        vision_model_use_patch_tokens=True,
        vision_model_num_head_blocks=2,
        vision_model_head_blocks_drop_path=0.3,
        vision_model_use_linear_projection=False,
        vision_model_patch_tokens_pooler_type="

In [3]:
import os
import sys
import json
import torch

DINOV3_REPO = '/home/ubuntu/youcook2-videos/repos/dinov3'
if DINOV3_REPO not in sys.path:
    sys.path.insert(0, DINOV3_REPO)

with open('/home/ubuntu/youcook2-videos/paths.json') as f:
    PATHS = json.load(f)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from dinov3.data.transforms import make_classification_eval_transform

# ── Update these two paths to your downloaded weight files ──────────────────
DINOV3_BACKBONE_WEIGHTS = '/home/ubuntu/weights/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth'
DINOV3_DINOTXT_WEIGHTS  = '/home/ubuntu/weights/dinov3_vitl16_dinotxt_vision_head_and_text_encoder-a442d8f5.pth'

for p, name in [(DINOV3_BACKBONE_WEIGHTS, 'backbone'), (DINOV3_DINOTXT_WEIGHTS, 'dinotxt')]:
    status = '✓' if os.path.exists(p) else '✗ NOT FOUND'
    print(f"{status}  {name}: {p}")

dinotxt, tokenizer = torch.hub.load(
    DINOV3_REPO,
    'dinov3_vitl16_dinotxt_tet1280d20h24l',
    source='local',
    weights=DINOV3_DINOTXT_WEIGHTS,
    backbone_weights=DINOV3_BACKBONE_WEIGHTS,
)
dinotxt = dinotxt.to(DEVICE).eval()

for p in dinotxt.parameters():
    p.requires_grad = False

SHARED_DIM     = dinotxt.model_config.embed_dim
CONTEXT_LENGTH = 77

trainable = sum(p.numel() for p in dinotxt.parameters() if p.requires_grad)
total     = sum(p.numel() for p in dinotxt.parameters())
print(f"Model loaded: {total/1e6:.1f}M parameters, {trainable} trainable (must be 0)")
assert trainable == 0, "Model must be fully frozen"
print(f"SHARED_DIM : {SHARED_DIM}")
print(f"All parameters frozen ✓")

# Save constants for NB5b to load without reloading the model
nb4b_constants = {
    'MODEL_NAME':     'dinov3_vitl16_dinotxt_tet1280d20h24l',
    'SHARED_DIM':     SHARED_DIM,
    'CONTEXT_LENGTH': CONTEXT_LENGTH,
}
with open(os.path.join(PATHS['processed'], 'nb4b_constants.json'), 'w') as f:
    json.dump(nb4b_constants, f, indent=2)
print(json.dumps(nb4b_constants, indent=2))

✓  backbone: /home/ubuntu/weights/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth
✓  dinotxt: /home/ubuntu/weights/dinov3_vitl16_dinotxt_vision_head_and_text_encoder-a442d8f5.pth
Model loaded: 866.6M parameters, 0 trainable (must be 0)
SHARED_DIM : 2048
All parameters frozen ✓
{
  "MODEL_NAME": "dinov3_vitl16_dinotxt_tet1280d20h24l",
  "SHARED_DIM": 2048,
  "CONTEXT_LENGTH": 77
}


## Image preprocessing

DINOv3 uses patch size 16 (vs DINOv2's patch size 14). The official
`make_classification_eval_transform()` from the DINOv3 repo handles the
correct resize/crop/normalisation for LVD-1689M pretrained models.
We print all transforms so the difference from NB4 is explicit.

In [4]:
from dinov3.data.transforms import make_classification_eval_transform

preprocess = make_classification_eval_transform()
print("Using official DINOv3 DinoTXT classification eval transform ✓")
for i, t in enumerate(preprocess.transforms):
    print(f"transforms[{i}]: {t}")

Using official DINOv3 DinoTXT classification eval transform ✓
transforms[0]: ToImage()
transforms[1]: Resize(size=[256], interpolation=InterpolationMode.BICUBIC, antialias=True)
transforms[2]: CenterCrop(size=(224, 224))
transforms[3]: Compose(
      ToDtype(scale=True)
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], inplace=False)
)


## Video embedding function

Identical logic to NB4:
```python
frames = extract_frames(video_path, start_sec, end_sec)  # up to 50 PIL images
frame_embeddings = dinotxt.encode_image(frame_batch)     # (n_frames, SHARED_DIM)
video_embedding  = frame_embeddings.mean(dim=0)          # (SHARED_DIM,)
```
`encode_image` runs the full frozen DinoTXT vision tower:
DINOv3 backbone → VisionHead → L2-normalised embedding in the shared space.

We process frames in sub-batches of 32 for A100 memory stability,
matching NB4's approach.

In [5]:
import numpy as np
from decord import VideoReader, cpu
from PIL import Image

MAX_FRAMES    = 50   # identical to NB4 — more than the paper's 8 because
                     # YouCook2 segments are longer and more fine-grained
FRAME_BATCH   = 32   # sub-batch size for GPU memory safety on A100

# Read IMAGE_SIZE from the official transform (CenterCrop size)
IMAGE_SIZE = preprocess.transforms[1].size[0]
print(f"IMAGE_SIZE from official transform: {IMAGE_SIZE}")


def extract_frames(video_path: str, start_sec: float, end_sec: float):
    """Extract up to MAX_FRAMES evenly spaced frames within the segment.
    Identical to NB4.
    """
    vr        = VideoReader(video_path, ctx=cpu(0))
    fps       = vr.get_avg_fps()
    start_idx = max(0, int(start_sec * fps))
    end_idx   = min(len(vr) - 1, int(end_sec * fps))
    if start_idx >= end_idx:
        return [Image.fromarray(vr[start_idx].asnumpy())]
    n_frames = min(MAX_FRAMES, end_idx - start_idx + 1)
    indices  = np.linspace(start_idx, end_idx, n_frames, dtype=int)
    frames   = vr.get_batch(indices).asnumpy()  # (n_frames, H, W, 3)
    return [Image.fromarray(frames[i]) for i in range(len(frames))]


@torch.no_grad()
def compute_video_embedding(video_path: str, start_sec: float, end_sec: float) -> torch.Tensor:
    """Runs the frozen DINOv3 DinoTXT vision tower on all frames and returns
    the mean-pooled embedding. Identical logic to NB4.

    encode_image handles: DINOv3 backbone → VisionHead → L2 normalisation
    Output shape: (SHARED_DIM,)
    """
    frames = extract_frames(video_path, start_sec, end_sec)

    # Stack frames: (n_frames, 3, IMAGE_SIZE, IMAGE_SIZE)
    frame_batch = torch.stack(
        [preprocess(frame) for frame in frames]
    ).to(DEVICE)

    # Process in sub-batches of FRAME_BATCH for memory safety
    all_embeddings = []
    with torch.autocast('cuda', dtype=torch.float):
        for i in range(0, len(frame_batch), FRAME_BATCH):
            sub = frame_batch[i:i + FRAME_BATCH]
            emb = dinotxt.encode_image(sub, normalize=True)  # (B, SHARED_DIM)
            all_embeddings.append(emb.cpu())

    # Mean pool over time dimension → (SHARED_DIM,)
    video_embedding = torch.cat(all_embeddings, dim=0).mean(dim=0)
    return video_embedding


# Shape and normalisation sanity check — identical to NB4
dummy_frames = torch.randn(4, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
with torch.no_grad():
    with torch.autocast('cuda', dtype=torch.float):
        test_emb = dinotxt.encode_image(dummy_frames, normalize=True)
print(f"encode_image output shape  : {test_emb.shape}  (expected: (4, {SHARED_DIM}))")
print(f"encode_image output L2 norm: {test_emb.norm(dim=-1).mean():.4f}  (expected: ~1.0)")
assert test_emb.shape == (4, SHARED_DIM)
assert abs(test_emb.norm(dim=-1).mean().item() - 1.0) < 0.01, "Output should be L2-normalised"
print("Shape and normalisation check ✓")

IMAGE_SIZE from official transform: 256
encode_image output shape  : torch.Size([4, 2048])  (expected: (4, 2048))
encode_image output L2 norm: 1.0000  (expected: ~1.0)
Shape and normalisation check ✓


## Compute and save DINOv3 video embeddings for all segments
Fully resumable — already computed files are skipped.

In [7]:
print(f"FRAME_BATCH : {FRAME_BATCH}")
print(f"IMAGE_SIZE  : {IMAGE_SIZE}")

# Check how many frames are being extracted per segment on average
sample_frames = extract_frames(
    os.path.join(VIDEO_DIR, f"{df.iloc[0]['video_id']}.mp4"),
    df.iloc[0]['start_sec'],
    df.iloc[0]['end_sec']
)
print(f"Frames per segment (sample) : {len(sample_frames)}")

# Check GPU memory
if torch.cuda.is_available():
    print(f"GPU memory free : {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")
    print(f"GPU memory total: {torch.cuda.mem_get_info()[1]/1e9:.1f} GB")

FRAME_BATCH : 32
IMAGE_SIZE  : 256
Frames per segment (sample) : 50
GPU memory free : 19.1 GB
GPU memory total: 23.7 GB


In [8]:
from tqdm.notebook import tqdm

EMB_DIR  = os.path.join(PATHS['embeddings'], 'dinov3')
VIDEO_DIR = PATHS['videos']
os.makedirs(EMB_DIR, exist_ok=True)

failed  = []
skipped = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc="Computing DINOv3 embeddings"):
    out_path = os.path.join(EMB_DIR, f"{row['segment_id']}.pt")

    # Skip if already computed
    if os.path.exists(out_path):
        skipped += 1
        continue

    video_path = os.path.join(VIDEO_DIR, f"{row['video_id']}.mp4")
    if not os.path.exists(video_path):
        failed.append({'segment_id': row['segment_id'], 'error': 'video not found'})
        continue

    try:
        emb = compute_video_embedding(video_path, row['start_sec'], row['end_sec'])
        torch.save(emb, out_path)
    except Exception as e:
        failed.append({'segment_id': row['segment_id'], 'error': str(e)})

saved = len([f for f in os.listdir(EMB_DIR) if f.endswith('.pt')])
print(f"Saved    : {saved:,}")
print(f"Skipped  : {skipped:,}")
print(f"Failed   : {len(failed):,}")
if failed:
    for f in failed[:10]:
        print(f"  {f['segment_id']}: {f['error']}")

Computing DINOv3 embeddings:   0%|          | 0/13029 [00:00<?, ?it/s]

[h264 @ 0x24f77900] mmco: unref short failure
[h264 @ 0x26614040] mmco: unref short failure
[h264 @ 0x26614040] mmco: unref short failure
[h264 @ 0x26614040] mmco: unref short failure
[h264 @ 0x26614040] mmco: unref short failure
[h264 @ 0x36fc1bc0] mmco: unref short failure
[h264 @ 0x36f3f340] mmco: unref short failure
[h264 @ 0x370031c0] mmco: unref short failure
[h264 @ 0x26614040] mmco: unref short failure
[h264 @ 0x24f77900] mmco: unref short failure
[h264 @ 0x38db5440] mmco: unref short failure
[h264 @ 0x38db5440] mmco: unref short failure
[h264 @ 0x42699fc0] mmco: unref short failure
[h264 @ 0x36f3f340] mmco: unref short failure
[h264 @ 0x24f3bbc0] mmco: unref short failure
[h264 @ 0x24f3bbc0] mmco: unref short failure
[h264 @ 0x24f77900] mmco: unref short failure
[h264 @ 0x26614040] mmco: unref short failure
[h264 @ 0x26614040] mmco: unref short failure
[h264 @ 0x36f3f340] mmco: unref short failure
[h264 @ 0x43093280] mmco: unref short failure
[h264 @ 0x43093280] mmco: unref sh

Saved    : 13,003
Skipped  : 485
Failed   : 26
  RubyHelAHBE_000: [00:47:16] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:104: Check failed: run_.load()
  RubyHelAHBE_001: [00:47:16] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:104: Check failed: run_.load()
  RubyHelAHBE_002: [00:47:16] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:104: Check failed: run_.load()
  RubyHelAHBE_003: [00:47:16] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:292: [00:47:16] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:218: Check failed: avcodec_send_packet(dec_ctx_.get(), pkt.get()) >= 0 (-11 vs. 0) Thread worker: Error sending packet.
  RubyHelAHBE_004: [00:47:16] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:104: Check failed: run_.load()
  RubyHelAHBE_005: [00:47:16] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:292: [00:47:16] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:218: Check failed: avcodec_send_packet(dec_ctx_.get(), pk

## Verify a sample embedding

In [9]:
sample_row  = df.sample(1).iloc[0]
sample_path = os.path.join(EMB_DIR, f"{sample_row['segment_id']}.pt")
emb         = torch.load(sample_path)

print(f"Segment  : {sample_row['segment_id']}")
print(f"Caption  : {sample_row['caption']}")
print(f"Shape    : {emb.shape}  (expected: torch.Size([{SHARED_DIM}]))")
print(f"L2 norm  : {emb.norm():.4f}  (expected: ~1.0)")
print(f"All zeros: {(emb == 0).all()}  (must be False)")

assert emb.shape == torch.Size([SHARED_DIM]), "Wrong embedding shape"
assert not (emb == 0).all(), "Embedding is all zeros"
print("\nEmbedding file looks correct ✓")

Segment  : SuK9T9KOBSs_002
Caption  : chop tomatos into small pieces and put in bowl
Shape    : torch.Size([2048])  (expected: torch.Size([2048]))
L2 norm  : 0.8078  (expected: ~1.0)
All zeros: False  (must be False)

Embedding file looks correct ✓


## Save clean CSV (excluding failed segments)
Mirrors the NB4 clean CSV step — produces a filtered dataframe for NB5b.

In [10]:
failed_ids    = {f['segment_id'] for f in failed}
failed_videos = {sid.rsplit('_', 1)[0] for sid in failed_ids}

if failed_ids:
    print(f"Failed segment IDs  : {failed_ids}")
    print(f"Failed video IDs    : {failed_videos}")

df_clean = df[~df['segment_id'].isin(failed_ids)].reset_index(drop=True)

print(f"\nOriginal segments : {len(df):,}")
print(f"Failed            : {len(failed_ids):,}")
print(f"Clean segments    : {len(df_clean):,}")

clean_path = '/home/ubuntu/youcook2-videos/annotations/segments_verbs_timed_dinov3_clean.csv'
df_clean.to_csv(clean_path, index=False)
print(f"Saved → {clean_path}")

Failed segment IDs  : {'oC5OvA4BK-E_008', 'RubyHelAHBE_004', 'RubyHelAHBE_002', 'r4MPTWizETs_002', 'oC5OvA4BK-E_010', 'r4MPTWizETs_004', 'RubyHelAHBE_005', 'oC5OvA4BK-E_006', 'RubyHelAHBE_000', 'oC5OvA4BK-E_011', 'oC5OvA4BK-E_012', 'RubyHelAHBE_003', 'oC5OvA4BK-E_007', 'oC5OvA4BK-E_002', 'oC5OvA4BK-E_009', 'oC5OvA4BK-E_001', 'r4MPTWizETs_005', 'r4MPTWizETs_006', 'r4MPTWizETs_003', 'r4MPTWizETs_001', 'RubyHelAHBE_001', 'oC5OvA4BK-E_004', 'oC5OvA4BK-E_000', 'oC5OvA4BK-E_005', 'oC5OvA4BK-E_003', 'r4MPTWizETs_000'}
Failed video IDs    : {'RubyHelAHBE', 'oC5OvA4BK-E', 'r4MPTWizETs'}

Original segments : 13,029
Failed            : 26
Clean segments    : 13,003
Saved → /home/ubuntu/youcook2-videos/annotations/segments_verbs_timed_dinov3_clean.csv


## Done ✓
Proceed to **Notebook 5b** (DINOv3 DinoTXT zero-shot verb prediction).

**What was produced:**
- `embeddings/dinov3/{segment_id}.pt` — (SHARED_DIM,) float32 tensor per segment
  Mean of `encode_image` outputs across up to 50 frames. L2-normalised.
- `processed/nb4b_constants.json` — SHARED_DIM, MODEL_NAME, CONTEXT_LENGTH

**Key points for NB5b:**
- Everything in DINOv3 DinoTXT is frozen — nothing trains in NB5b
- Text embeddings are computed on-the-fly in NB5b (same as NB5)
- The embedding dimension and shared space are the same as NB4/NB5
  (both use `tet1280d20h24l` — 1280-dim text encoder tower)
- Results are directly comparable to NB5 (DINOv2) output